# Numerical Computing with NumPy

## The Dataset: UCI Wine Quality

**1. Load a Dataset Directly from a GitHub Repository**

In [1]:
import numpy as np

url = (
    "https://raw.githubusercontent.com/mlflow/mlflow/"
    "master/tests/datasets/winequality-red.csv"
)

# Load CSV — skip the header row, use semicolon delimiter
data = np.genfromtxt(url, delimiter=";", skip_header=1)

print(data.shape)   # (1599, 12)
print(data.dtype)   # float64

(1599, 12)
float64


**2. Load a Dataset Locally**

In [3]:
import numpy as np

path = "../data/winequality-red.csv"

# Load CSV — skip the header row, use semicolon delimiter
data = np.genfromtxt(path, delimiter=";", skip_header=1)

print(data.shape)   # (1599, 12)
print(data.dtype)   # float64

(1599, 12)
float64


In [ ]:
## Arrays and Vectorized Operations

### What is an ndarray?

In [4]:
import numpy as np

# A simple 1-D array
arr = np.array([7.4, 7.8, 7.8, 11.2, 7.4])
print(arr.dtype)    # float64
print(arr.shape)    # (5,)
print(arr.ndim)     # 1
print(arr.size)     # 5

float64
(5,)
1
5


In [5]:
# A 2-D array (matrix) is created from a list of lists:

matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
print(matrix.shape)   # (2, 3) — 2 rows, 3 columns

(2, 3)


### Array creation

In [6]:
np.zeros((3, 4))          # 3×4 array of zeros
np.ones((2, 5))           # 2×5 array of ones
np.eye(4)                 # 4×4 identity matrix
np.arange(0, 10, 2)       # [0, 2, 4, 6, 8]
np.linspace(0, 1, 5)      # [0.0, 0.25, 0.5, 0.75, 1.0]
np.random.seed(42)
np.random.randn(3, 3)     # 3×3 standard normal random array

array([[ 0.49671415, -0.1382643 ,  0.64768854],
       [ 1.52302986, -0.23415337, -0.23413696],
       [ 1.57921282,  0.76743473, -0.46947439]])

### Indexing and slicing

In [7]:
# Extract the alcohol column (index 10) from the wine dataset
alcohol = data[:, 10]      # all rows, column 10
print(alcohol[:5])         # first 5 values: [9.4 9.8 9.8 9.8 9.4]

# Extract the first 5 rows and first 3 columns
subset = data[:5, :3]
print(subset)

[9.4 9.8 9.8 9.8 9.4]
[[ 7.4   0.7   0.  ]
 [ 7.8   0.88  0.  ]
 [ 7.8   0.76  0.04]
 [11.2   0.28  0.56]
 [ 7.4   0.7   0.  ]]


Boolean indexing filters rows by a condition:

In [8]:
# Select wines with alcohol content above 12%
high_alcohol = data[data[:, 10] > 12]
print(high_alcohol.shape)  # (141, 12)

(141, 12)


In [ ]:
Fancy indexing uses an array of indices to select non-contiguous columns:

In [9]:
# Extract alcohol (10), pH (8), and quality (11) columns
cols = [10, 8, 11]
selected = data[:, cols]
print(selected.shape)  # (1599, 3)

(1599, 3)


### Vectorized arithmetic

In [10]:
alcohol = data[:, 10]   # alcohol % vol
density = data[:, 7]    # density g/cm³

# Compute alcohol-to-density ratio for every wine sample at once
ratio = alcohol / density
print(ratio[:5])

[9.4207256  9.83146067 9.82948847 9.81963928 9.4207256 ]


In [11]:
# Log-transform residual sugar (column 3) to reduce skewness
sugar = data[:, 3]
log_sugar = np.log1p(sugar)   # log(1 + x) — safe for zero values

# Standardise alcohol: z = (x - mean) / std
alcohol_z = (alcohol - alcohol.mean()) / alcohol.std()
print(alcohol_z[:5])

[-0.96024611 -0.58477711 -0.58477711 -0.58477711 -0.96024611]


### Aggregations

In [12]:
quality = data[:, 11]

print(f"Mean quality:   {quality.mean():.2f}")
print(f"Std quality:    {quality.std():.2f}")
print(f"Min quality:    {quality.min():.0f}")
print(f"Max quality:    {quality.max():.0f}")
print(f"Median quality: {np.median(quality):.1f}")

Mean quality:   5.64
Std quality:    0.81
Min quality:    3
Max quality:    8
Median quality: 6.0


For 2-D arrays

In [14]:
# axis=0 → aggregate across rows (per column)
col_means = data.mean(axis=0)   # shape (12,) — mean of each feature
print(col_means)

# axis=1 → aggregate across columns (per row)
row_means = data.mean(axis=1)   # shape (1599,) — mean of each sample
print(row_means)

[ 8.31963727  0.52782051  0.27097561  2.5388055   0.08746654 15.87492183
 46.46779237  0.99674668  3.3111132   0.65814884 10.42298311  5.63602251]
[ 6.21198333 10.25456667  8.30825    ...  8.37347833  8.76795583
  7.7077075 ]


## Efficient Numerical Computation

### Broadcasting

In [15]:
# col_means has shape (12,)
# data has shape (1599, 12)
# Broadcasting: col_means is treated as (1, 12) and broadcast over all rows
data_centred = data - col_means
print(data_centred.shape)   # (1599, 12)
print(data_centred.mean(axis=0).round(10))  # ~0 for all columns

(1599, 12)
[-0. -0.  0. -0.  0.  0. -0.  0. -0. -0.  0. -0.]


Standardisation:

In [16]:
col_stds = data.std(axis=0)
data_scaled = (data - col_means) / col_stds
print(data_scaled.mean(axis=0).round(8))   # all ~0
print(data_scaled.std(axis=0).round(8))    # all ~1

[-0. -0.  0. -0.  0.  0. -0.  0. -0. -0.  0. -0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Boolean masks and filtering

In [17]:
quality = data[:, 11]
alcohol = data[:, 10]

# High-quality wines: quality >= 7
high_q = quality >= 7

# Low-alcohol wines: alcohol <= 10
low_alc = alcohol <= 10.0

# High quality AND low alcohol
mask = high_q & low_alc
print(data[mask].shape)       # number of wines meeting both criteria
print(f"Count: {mask.sum()}")

(21, 12)
Count: 21


`np.where`:

In [18]:
# Label wines as 'good' (1) or 'average' (0) based on quality
labels = np.where(quality >= 7, 1, 0)
print(np.unique(labels, return_counts=True))

(array([0, 1]), array([1382,  217]))


### Performance: vectorization vs. loops

In [19]:
import time

# Loop version
start = time.time()
norms_loop = []
for row in data:
    norms_loop.append(np.sqrt(np.sum(row**2)))
loop_time = time.time() - start

# Vectorized version
start = time.time()
norms_vec = np.sqrt((data**2).sum(axis=1))
vec_time = time.time() - start

print(f"Loop:       {loop_time * 1000:.2f} ms")
print(f"Vectorized: {vec_time * 1000:.2f} ms")
print(f"Speedup:    {loop_time / vec_time:.1f}×")

Loop:       19.80 ms
Vectorized: 0.36 ms
Speedup:    55.4×


### Memory layout: views vs. copies

In [20]:
# Slicing → view (modifying slice modifies original)
alcohol_view = data[:, 10]
alcohol_view[0] = 999          # modifies data[0, 10]

# Boolean indexing → copy (safe to modify)
high_q_data = data[data[:, 11] >= 7].copy()
high_q_data[:, 10] = 0        # does NOT affect data

In [ ]:
## Matrix Operations

### The dot product

In [21]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print(np.dot(a, b))    # 1×4 + 2×5 + 3×6 = 32
print(a @ b)           # equivalent using @ operator (Python 3.5+)

32
32


### Matrix multiplication

In [22]:
# data_scaled: shape (1599, 12)
# Transpose: shape (12, 1599)
# Result: (12, 12) — feature cross-product matrix
X = data_scaled[:, :11]   # features only (exclude quality)
XtX = X.T @ X            # shape (11, 11)
print(XtX.shape)
print(XtX.diagonal()[:5].round(2))  # diagonal ≈ n-1 for standardised data

(11, 11)
[1599. 1599. 1599. 1599. 1599.]


### Transpose

In [23]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

print(A.T)

print(A.shape)    
print(A.T.shape) 

[[1 4]
 [2 5]
 [3 6]]
(2, 3)
(3, 2)


### Solving a linear system

In [24]:
from numpy.linalg import lstsq

X = data_scaled[:, :11]          # features: (1599, 11)
y = data_scaled[:, 11]           # quality: (1599,)

# Add intercept column (column of ones)
X_bias = np.column_stack([np.ones(len(X)), X])   # (1599, 12)

# Solve: minimise ||X_bias @ coef - y||²
coef, residuals, rank, sv = lstsq(X_bias, y, rcond=None)

print("Intercept:        ", round(coef[0], 4))
print("Alcohol coef:     ", round(coef[11], 4))   # last feature = alcohol
print("Volatile acid:    ", round(coef[2], 4))

Intercept:         0.0
Alcohol coef:      0.3645
Volatile acid:     -0.2403


### Eigenvalues and PCA by hand

In [25]:
from numpy.linalg import eig, eigh

X = data_scaled[:, :11]
cov = (X.T @ X) / (len(X) - 1)     # (11, 11) covariance matrix

# eigh is preferred for symmetric matrices (more numerically stable)
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# Sort by descending eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Variance explained by each principal component
var_explained = eigenvalues / eigenvalues.sum()
print("Variance explained (first 3 PCs):")
for i, v in enumerate(var_explained[:3]):
    print(f"  PC{i+1}: {v*100:.1f}%")

Variance explained (first 3 PCs):
  PC1: 28.2%
  PC2: 17.5%
  PC3: 14.1%


### Singular Value Decomposition (SVD)

In [26]:
from numpy.linalg import svd

X = data_scaled[:, :11]
U, S, Vt = svd(X, full_matrices=False)

print(f"U:  {U.shape}")    # (1599, 11)
print(f"S:  {S.shape}")    # (11,)  — singular values
print(f"Vt: {Vt.shape}")   # (11, 11)

# Project data onto the first 2 principal components
X_pca = X @ Vt[:2].T      # shape (1599, 2)
print(X_pca.shape)

U:  (1599, 11)
S:  (11,)
Vt: (11, 11)
(1599, 2)
